# Assignment 2C — Part B: Custom Prompt Chains
Implements the Verification chain and Socratic chain, prints intermediate reasoning artifacts required by the brief, and compares both chains with RAG alone on five targeted question types.


In [1]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
print("Project:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())

Project: /content/banking-compliance-llm-assignment-1a
src exists: True


In [2]:
!ls

'Assignment Documentation'		     notebooks	        results
 Banking-compliance-llm-assignment-1a.docx   README.md	        run_pipeline.py
 data					     reports	        src
 models					     requirements.txt


In [3]:
!python -m pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 38.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 45.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.9/947.9 kB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# This cell installs the model inference, PEFT, embedding, and FAISS retrieval dependencies.%pip install -q transformers peft accelerate sentence-transformers faiss-cpu

In [4]:
# This cell loads the Assignment 1 model or a configurable Hugging Face fallback, reads the first ten benchmark examples, and builds a FAISS index over cleaned domain text.
from pathlib import Path
import os, json, time, re, warnings
import torch, numpy as np, pandas as pd, faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("/content/LLM_Project_1_2")
DATA = ROOT / "data"
MODEL_ID = os.getenv("MODEL_ID", "HuggingFaceTB/SmolLM2-360M")
BASE_PATH = os.getenv("BASE_MODEL_DIR", str(ROOT / "models" / "base_model"))
ADAPTER_PATH = os.getenv("ADAPTER_DIR", str(ROOT / "models" / "adapter_A"))
if not Path(BASE_PATH, "config.json").exists():
    BASE_PATH = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)
model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
adapter_loaded = Path(ADAPTER_PATH, "adapter_config.json").exists()
if adapter_loaded:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    warnings.warn("Assignment 1 adapter not found; using base model. Set ADAPTER_DIR to your uploaded adapter for required final results.")
model.eval()

def generate(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, int(output.shape[-1]), time.perf_counter() - started

dataset_path = DATA / "instruction_dataset" / "instruction_dataset.jsonl"
if not dataset_path.exists():
    raise FileNotFoundError(f"Expected benchmark file: {dataset_path}")
benchmark = []
with dataset_path.open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            row = json.loads(line)
            benchmark.append({
                "query": row.get("query", row.get("instruction", row.get("question", ""))),
                "answer": row.get("answer", row.get("response", ""))
            })
        if len(benchmark) == 10:
            break
if len(benchmark) < 10:
    raise ValueError("The benchmark file must contain at least ten question-answer records.")

def read_chunks(folder=DATA / "cleaned_txt", chunk_words=220, overlap=40):
    chunks = []
    step = chunk_words - overlap
    for file in folder.rglob("*.txt"):
        words = file.read_text(encoding="utf-8", errors="ignore").split()
        for start in range(0, len(words), step):
            text = " ".join(words[start:start + chunk_words])
            if text:
                chunks.append({"source": str(file.relative_to(folder)), "text": text})
    if not chunks:
        raise FileNotFoundError(f"No cleaned .txt corpus found under {folder}")
    return chunks

chunks = read_chunks()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode([item["text"] for item in chunks], normalize_embeddings=True, show_progress_bar=True).astype("float32")
faiss_index = faiss.IndexFlatIP(vectors.shape[1])
faiss_index.add(vectors)

def retrieve(query, k=2):
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = faiss_index.search(query_vector, min(k, len(chunks)))
    return [{"score": float(score), **chunks[idx]} for score, idx in zip(scores[0], ids[0]) if idx >= 0]

print(f"Questions: {len(benchmark)} | text chunks: {len(chunks)} | GPU: {torch.cuda.is_available()} | adapter loaded: {adapter_loaded}")



config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

... (17 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/314 [00:00<?, ?it/s]

Questions: 10 | text chunks: 10044 | GPU: False | adapter loaded: False


In [5]:
# This cell implements the four required base patterns with comparable answer, token, and latency outputs. RAG also returns its retrieved chunks for inspection.
def zero_shot(query):
    return generate(query)

def few_shot(query, examples=None, k=3):
    examples = (examples or benchmark)[:k]
    demonstrations = "\n\n".join(
        f"Question: {item['query']}\nAnswer: {item['answer']}" for item in examples
    )
    prompt = f"Use these worked examples as guidance.\n{demonstrations}\n\nQuestion: {query}\nAnswer:"
    return generate(prompt)

def rag(query):
    hits = retrieve(query, k=2)
    context = "\n\n".join(f"[{item['source']}] {item['text']}" for item in hits)
    prompt = f"Answer using only the evidence below. If it is insufficient, say so.\nEvidence:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generate(prompt)
    return result, hits

def self_critique(query):
    draft, tokens1, latency1 = generate(f"Answer the question accurately: {query}")
    critique, tokens2, latency2 = generate(
        f"Check the draft for errors, missing points, and unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, tokens3, latency3 = generate(
        f"Revise the draft using the critique. Do not add unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique: {critique}\nRevised answer:"
    )
    return revised, tokens1 + tokens2 + tokens3, latency1 + latency2 + latency3, draft, critique

def accuracy_score(answer, reference):
    # This overlap estimate is an aid only; manually review and assign the official 0–2 rubric.
    answer_words = set(re.findall(r"\w+", answer.lower()))
    reference_words = set(re.findall(r"\w+", reference.lower()))
    overlap = len(answer_words & reference_words) / max(1, len(reference_words))
    return 2 if overlap >= 0.65 else (1 if overlap >= 0.30 else 0)



In [6]:
# This cell defines the Verification and Socratic chains and tests them against RAG on five targeted banking/regulatory query types.
def verification_chain(query):
    hits = retrieve(query, k=2)
    evidence = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    draft, t1, l1 = generate(f"Answer from the supplied evidence only.\nEvidence:\n{evidence}\nQuestion: {query}\nDraft:")
    critique, t2, l2 = generate(
        f"Critique domain accuracy against the evidence. Identify unsupported claims.\nEvidence:\n{evidence}\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, t3, l3 = generate(
        f"Revise using the evidence and critique; do not invent facts.\nEvidence:\n{evidence}\nDraft: {draft}\nCritique: {critique}\nFinal:"
    )
    return {"answer": revised, "draft": draft, "critique": critique, "observations": hits,
            "tokens": t1+t2+t3, "latency_sec": l1+l2+l3}

def socratic_chain(query):
    decomposition, t0, l0 = generate(
        f"Break the question into exactly two independent, answerable sub-questions. Return one per line.\nQuestion: {query}"
    )
    sub_questions = [line.strip(" -*0123456789.\t") for line in decomposition.splitlines() if line.strip()][:2]
    if len(sub_questions) < 2:
        sub_questions = [query, query]
    observations, total_tokens, total_latency = [], t0, l0
    for sub_question in sub_questions:
        (answer, tokens, latency), hits = rag(sub_question)
        observations.append({"sub_question": sub_question, "answer": answer, "retrieved": hits})
        total_tokens += tokens
        total_latency += latency
    synthesis, tokens, latency = generate(
        f"Answer the original question using only these per-question findings.\nQuestion: {query}\nFindings: {observations}\nFinal answer:"
    )
    return {"answer": synthesis, "sub_questions": sub_questions, "observations": observations,
            "tokens": total_tokens+tokens, "latency_sec": total_latency+latency}

targeted_queries = [
    ("Factual + verification", "According to the corpus, what is the required record-retention period for this compliance process?"),
    ("Compliance check", "What evidence should an analyst verify before concluding that a customer onboarding control is compliant?"),
    ("Multi-part process", "What steps should be followed for a suspicious transaction and which roles are responsible?"),
    ("Two regulations", "Compare the relevant RBI and SEBI requirements for reporting a suspected financial crime."),
    ("Ambiguous domain-specific", "Does the exception apply in this case, and what facts are needed to decide?")
]
chain_rows = []
for query_type, query in targeted_queries:
    verify = verification_chain(query)
    socratic = socratic_chain(query)
    (rag_answer, rag_tokens, rag_latency), rag_hits = rag(query)
    chain_rows.append({
        "query_type": query_type, "query": query,
        "verification_answer": verify["answer"], "verification_tokens": verify["tokens"],
        "verification_draft": verify["draft"], "verification_critique": verify["critique"],
        "socratic_answer": socratic["answer"], "socratic_tokens": socratic["tokens"],
        "socratic_subquestions": socratic["sub_questions"], "socratic_observations": socratic["observations"],
        "rag_answer": rag_answer, "rag_tokens": rag_tokens, "rag_retrieved": rag_hits,
        "human_best_choice": None, "human_accuracy_verification_0_2": None,
        "human_accuracy_socratic_0_2": None, "human_accuracy_rag_0_2": None
    })
part_b_results = pd.DataFrame(chain_rows)
display(part_b_results)



,query_type,query,verification_answer,verification_tokens,verification_draft,verification_critique,socratic_answer,socratic_tokens,socratic_subquestions,socratic_observations,rag_answer,rag_tokens,rag_retrieved,human_best_choice,human_accuracy_verification_0_2,human_accuracy_socratic_0_2,human_accuracy_rag_0_2
0,Factual + verification,"According to the corpus, what is the required ...",19.1.22. The record-keeping period shall be fi...,3469,19.1.1. The record-keeping period shall be fiv...,19.1.12. The record-keeping period shall be fi...,"Question: According to the corpus, what is the...",4059,"[Answer: 12 months, Question: What is the requ...","[{'sub_question': 'Answer: 12 months', 'answer...",The required record-retention period is 5 year...,908,"[{'score': 0.5692165493965149, 'source': 'IRDA...",None,None,None,None
1,Compliance check,What evidence should an analyst verify before ...,[SEBI/1769776024792.txt] • Verify that the cus...,3373,[SEBI/1769776024792.txt] • Obtain an understan...,[SEBI/1769776024792.txt] • Verify that the cus...,Question: What evidence should an analyst veri...,4692,[Answer: The customer onboarding control is co...,[{'sub_question': 'Answer: The customer onboar...,The customer onboarding control is a control t...,877,"[{'score': 0.4417920708656311, 'source': 'SEBI...",None,None,None,None
2,Multi-part process,What steps should be followed for a suspicious...,114\n\n[Page 41]\n\n[Page 42]\n\n[Page 43]\n\n...,3799,111\n\n[Page 14] 112 (b) Transactions involvin...,[RBI/MD123C0C6F7FF70E34F3DA5623B9E72AB3181.txt...,Question: What is the current status of the tr...,4956,[Question: What is the current status of the t...,[{'sub_question': 'Question: What is the curre...,The PSO shall monitor suspicious transactions ...,1017,"[{'score': 0.5240437984466553, 'source': 'RBI/...",None,None,None,None
3,Two regulations,Compare the relevant RBI and SEBI requirements...,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,3670,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,Question: Compare the relevant RBI and SEBI re...,4278,[Question: What are the key differences betwee...,[{'sub_question': 'Question: What are the key ...,The SEBI and RBI have different requirements f...,974,"[{'score': 0.6679743528366089, 'source': 'RBI/...",None,None,None,None
4,Ambiguous domain-specific,"Does the exception apply in this case, and wha...",1786957172778.txt\n\n[SEBI/1786957172778.txt] ...,3462,1786957172778.txt\n\n[SEBI/1786957172778.txt] ...,[SEBI/1786957172778.txt] (9) the provisions of...,Question: Does the exception apply in this cas...,4000,"[The first question is:, Does the exception ap...","[{'sub_question': 'The first question is:', 'a...",No. The Board has not made any specific provis...,905,"[{'score': 0.4312416911125183, 'source': 'SEBI...",None,None,None,None


## Required 75-word analysis
Complete human accuracy scores and write 75 words on which query types benefited from chaining over RAG and whether the extra token cost was justified.
